In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

project5sanju_features_for_svm_path = kagglehub.dataset_download('project5sanju/features-for-svm')
project5sanju_gan_training_data_path = kagglehub.dataset_download('project5sanju/gan-training-data')
project5sanju_gan_second_adv_feature_path = kagglehub.dataset_download('project5sanju/gan-second-adv-feature')
project5sanju_mc_test_path = kagglehub.dataset_download('project5sanju/mc-test')
project5sanju_normal_data_path = kagglehub.dataset_download('project5sanju/normal-data')
project5sanju_normal_train_dataset_path = kagglehub.dataset_download('project5sanju/normal-train-dataset')
project5sanju_test_data_for_all_path = kagglehub.dataset_download('project5sanju/test-data-for-all')

print('Data source import complete.')


CNN_FILES_FINAL

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


dataset_path = "/kaggle/input/datasets/project5sanju/normal-train-dataset/RESIZED ALL"

class_names = sorted(os.listdir(dataset_path))
print("Classes:", class_names)


image_paths = []
labels = []

for idx, class_name in enumerate(class_names):

    class_folder = os.path.join(dataset_path, class_name)
    for image_name in os.listdir(class_folder):
        image_paths.append(os.path.join(class_folder, image_name))
        labels.append(idx)



train_paths, val_paths, train_labels, val_labels = train_test_split(image_paths,labels,test_size=0.20,stratify=labels,random_state=42)

print("Train size:", len(train_paths))
print("Val size:", len(val_paths))



train_transform = transforms.Compose([transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5],std=[0.5,0.5,0.5])])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5],std=[0.5,0.5,0.5])])


class SkinDataset(Dataset):

    def __init__(self, image_paths, labels, transform=None):

        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(self.image_paths[idx]).convert("RGB")

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label, self.image_paths[idx]


train_dataset = SkinDataset(train_paths,train_labels,train_transform)
val_dataset = SkinDataset(val_paths,val_labels,val_transform)


train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False,num_workers=2)


class_counts = np.bincount(train_labels)
print("Class counts:", class_counts)
weights = 1.0 / np.sqrt(class_counts)
weights = weights / weights.sum()
class_weights = torch.tensor(weights,dtype=torch.float).to(device)



class CustomCNN(nn.Module):

    def __init__(self, num_classes):

        super(CustomCNN, self).__init__()


        self.features = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1,1)))

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(256, 512),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(512,num_classes))

    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)

        return x



model = CustomCNN(len(class_names)).to(device)


criterion = nn.CrossEntropyLoss(weight=class_weights,label_smoothing=0.1)

optimizer = optim.AdamW(model.parameters(),lr=0.0003,weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=0.5,patience=4)


best_val_acc = 0
best_epoch = 0

patience = 10
counter = 0

num_epochs = 45
train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []


for epoch in range(num_epochs):

    model.train()

    train_loss = 0
    correct = 0
    total = 0

    for images, labels, paths in train_loader:

        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=2.0)

        optimizer.step()
        train_loss += loss.item()
        _, preds = torch.max(outputs,1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_acc = correct / total
    train_losses.append(train_loss / len(train_loader))
    train_accuracies.append(train_acc)



    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    val_true = []
    val_pred = []

    with torch.no_grad():

        for images, labels, paths in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs,1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

            val_true.extend(labels.cpu().numpy())
            val_pred.extend(preds.cpu().numpy())

    val_acc = correct / total
    val_losses.append(val_loss / len(val_loader))
    val_accuracies.append(val_acc)
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Acc: {train_acc:.4f} "
        f"Val Acc: {val_acc:.4f} "
        f"LR: {current_lr:.6f}")


    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch + 1

        torch.save(model.state_dict(),"best_model.pth")

        counter = 0
        print("✅ Best model updated!")

    else:
        counter += 1

    if counter >= patience:
        print("\nEarly stopping triggered!")
        break


model.load_state_dict(torch.load("best_model.pth"))

print("\n===== TRAINING COMPLETE =====")
print("Best Validation Accuracy:", best_val_acc)
print("Best Epoch:", best_epoch)

Device: cuda
Classes: ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
Train size: 6741
Val size: 1686
Class counts: [ 524  328  700  147  705 4176  161]
Epoch [1/45] Train Acc: 0.5820 Val Acc: 0.6684 LR: 0.000300
✅ Best model updated!
Epoch [2/45] Train Acc: 0.6204 Val Acc: 0.6483 LR: 0.000300
Epoch [3/45] Train Acc: 0.6315 Val Acc: 0.7088 LR: 0.000300
✅ Best model updated!
Epoch [4/45] Train Acc: 0.6539 Val Acc: 0.6957 LR: 0.000300
Epoch [5/45] Train Acc: 0.6545 Val Acc: 0.6560 LR: 0.000300
Epoch [6/45] Train Acc: 0.6729 Val Acc: 0.6287 LR: 0.000300
Epoch [7/45] Train Acc: 0.6750 Val Acc: 0.7260 LR: 0.000300
✅ Best model updated!
Epoch [8/45] Train Acc: 0.6831 Val Acc: 0.6649 LR: 0.000300
Epoch [9/45] Train Acc: 0.6883 Val Acc: 0.6880 LR: 0.000300
Epoch [10/45] Train Acc: 0.6999 Val Acc: 0.6595 LR: 0.000300
Epoch [11/45] Train Acc: 0.6922 Val Acc: 0.7278 LR: 0.000300
✅ Best model updated!
Epoch [12/45] Train Acc: 0.7002 Val Acc: 0.7242 LR: 0.000300
Epoch [13/45] Train Acc: 0.7081 V

In [ ]:
model.eval()

val_true = []
val_pred = []

with torch.no_grad():

    for images, labels, paths in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        val_true.extend(labels.numpy())
        val_pred.extend(preds.cpu().numpy())

print("\n===== VALIDATION CLASSIFICATION REPORT =====\n")

print(classification_report(val_true,val_pred,target_names=class_names))


val_cm = confusion_matrix(val_true,val_pred)
val_cm_df = pd.DataFrame(val_cm,
    index=[f"True_{name}" for name in class_names],
    columns=[f"Pred_{name}" for name in class_names])

print("\n===== VALIDATION CONFUSION MATRIX =====\n")
print(val_cm_df)


train_true = []
train_pred = []

with torch.no_grad():

    for images, labels, paths in train_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        train_true.extend(labels.numpy())
        train_pred.extend(preds.cpu().numpy())

print("\n===== TRAIN CLASSIFICATION REPORT =====\n")

print(classification_report(train_true,train_pred,target_names=class_names))


train_cm = confusion_matrix(train_true,train_pred)
train_cm_df = pd.DataFrame(train_cm,
    index=[f"True_{name}" for name in class_names],
    columns=[f"Pred_{name}" for name in class_names])

print("\n===== TRAIN CONFUSION MATRIX =====\n")
print(train_cm_df)


===== VALIDATION CLASSIFICATION REPORT =====

              precision    recall  f1-score   support

       AKIEC       0.83      0.73      0.78       131
         BCC       0.59      0.61      0.60        82
         BKL       0.68      0.54      0.60       175
          DF       0.37      0.54      0.44        37
         MEL       0.44      0.61      0.51       176
          NV       0.90      0.87      0.88      1044
        VASC       0.72      0.68      0.70        41

    accuracy                           0.77      1686
   macro avg       0.65      0.65      0.64      1686
weighted avg       0.79      0.77      0.78      1686


===== VALIDATION CONFUSION MATRIX =====

            Pred_AKIEC  Pred_BCC  Pred_BKL  Pred_DF  Pred_MEL  Pred_NV  \
True_AKIEC          96         9         7       12         4        1   
True_BCC             2        50         9        3        11        6   
True_BKL             5         6        94        7        29       34   
True_DF           

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

CustomCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [ ]:
test_dir = "/kaggle/input/datasets/project5sanju/test-data-for-all/RESIZED_TESTDATA"   # ✅ ADD THIS

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

test_transform = transforms.Compose([
    transforms.Resize((244,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5],std=[0.5,0.5,0.5])])
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)



In [ ]:

from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [ ]:
class_names = test_dataset.classes

print("\n Unseen Classification Report:\n")                                                           # UNSEEN DATA
print(classification_report(all_labels, all_preds, target_names=class_names))


 Unseen Classification Report:

              precision    recall  f1-score   support

       AKIEC       0.67      0.79      0.73       164
         BCC       0.51      0.67      0.58       103
         BKL       0.60      0.38      0.46       219
          DF       0.35      0.28      0.31        47
         MEL       0.44      0.62      0.51       221
          NV       0.91      0.85      0.88      1306
        VASC       0.64      0.86      0.73        51

    accuracy                           0.75      2111
   macro avg       0.59      0.64      0.60      2111
weighted avg       0.77      0.75      0.75      2111



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(all_labels, all_preds)
cm_df = pd.DataFrame(cm,
    index=[f"True_{name}" for name in class_names],
    columns=[f"Pred_{name}" for name in class_names])

print("\n===== CONFUSION MATRIX =====\n")

print(cm_df)


===== CONFUSION MATRIX =====

            Pred_AKIEC  Pred_BCC  Pred_BKL  Pred_DF  Pred_MEL  Pred_NV  \
True_AKIEC         130         8         8        5         4        5   
True_BCC            10        69         4        2         5        9   
True_BKL            22        18        83        5        53       35   
True_DF              9         9         2       13         4        7   
True_MEL             8         4        13        0       136       56   
True_NV             15        26        28       10       108     1112   
True_VASC            0         1         0        2         1        3   

            Pred_VASC  
True_AKIEC          4  
True_BCC            4  
True_BKL            3  
True_DF             3  
True_MEL            4  
True_NV             7  
True_VASC          44  


In [ ]:


import os
import pandas as pd
import numpy as np

from sklearn.metrics import (classification_report,confusion_matrix,accuracy_score,precision_score,recall_score,f1_score)


output_folder = "/kaggle/working/cnn_results"
os.makedirs(output_folder, exist_ok=True)

print("Saving files to:")
print(output_folder)


image_names = [os.path.basename(path)
    for path, _ in test_dataset.samples]

predictions_df = pd.DataFrame({
    "image_name": image_names,
    "true_label": all_labels,
    "predicted_label": all_preds})

predictions_df.to_csv(os.path.join(output_folder, "predictions.csv"),index=False)


test_report_dict = classification_report(all_labels,all_preds,target_names=class_names,output_dict=True)

test_report_df = pd.DataFrame(test_report_dict).transpose()

test_report_df.to_csv(os.path.join(output_folder, "test_classification_report.csv"))


test_cm = confusion_matrix(all_labels,all_preds)

test_cm_df = pd.DataFrame(test_cm,
    index=[f"True_{name}" for name in class_names],
    columns=[f"Pred_{name}" for name in class_names])

test_cm_df.to_csv(os.path.join(output_folder, "test_confusion_matrix.csv"))


train_report_dict = classification_report(train_true,train_pred,target_names=class_names,output_dict=True)
train_report_df = pd.DataFrame(train_report_dict).transpose()
train_report_df.to_csv(os.path.join(output_folder, "train_classification_report.csv"))
train_cm_df.to_csv(os.path.join(output_folder, "train_confusion_matrix.csv"))


val_report_dict = classification_report(val_true,val_pred,target_names=class_names,output_dict=True)
val_report_df = pd.DataFrame(val_report_dict).transpose()
val_report_df.to_csv(os.path.join(output_folder, "validation_classification_report.csv"))


val_cm_df.to_csv(os.path.join(output_folder, "validation_confusion_matrix.csv"))


try:

    history_df = pd.DataFrame({"epoch": range(1, len(train_losses)+1),
        "train_loss": train_losses,
        "val_loss": val_losses,
        "train_accuracy": train_accuracies,
        "val_accuracy": val_accuracies})

    history_df.to_csv(os.path.join(output_folder, "epoch_history.csv"),index=False)

except:
    print("Epoch history lists not found")


summary_df = pd.DataFrame({"metric": ["best_validation_accuracy","best_epoch","test_accuracy","weighted_precision","weighted_recall","weighted_f1_score"],
    "value": [best_val_acc,best_epoch,accuracy_score(all_labels, all_preds),
              precision_score(all_labels,all_preds,average="weighted"),
              recall_score(all_labels,all_preds,average="weighted"),
              f1_score(all_labels,all_preds,average="weighted")]})

summary_df.to_csv(os.path.join(output_folder, "summary_metrics.csv"),index=False)


class_distribution_df = pd.DataFrame({"class_name": class_names,"train_count": np.bincount(train_labels),"validation_count": np.bincount(val_labels),"test_count": np.bincount(all_labels)})
class_distribution_df.to_csv(os.path.join(output_folder, "class_distribution.csv"),index=False)


split_df = pd.DataFrame({"split": ["train","validation","test"],
    "count": [len(train_dataset),len(val_dataset),len(test_dataset)]})

split_df.to_csv(os.path.join(output_folder, "dataset_split_counts.csv"),index=False)



print("\nALL FILES SAVED SUCCESSFULLY")
print("\nSaved files:")
for file in os.listdir(output_folder):
    print(file)

Saving files to:
/kaggle/working/cnn_results

ALL FILES SAVED SUCCESSFULLY

Saved files:
train_confusion_matrix.csv
epoch_history.csv
validation_confusion_matrix.csv
test_classification_report.csv
test_confusion_matrix.csv
validation_classification_report.csv
summary_metrics.csv
predictions.csv
dataset_split_counts.csv
class_distribution.csv
train_classification_report.csv
